In [0]:
%pip install pandas pyreadstat

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 57.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyreadstat]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import pyreadstat
from pathlib import Path
import re

# ----------------------------------------
# 1. Define source and destination
# ----------------------------------------

source_folder = Path(
    "/Volumes/nhanes_2017___2018/nhanes_bronze/raw_xpt_files"
)

catalog = "nhanes_2017___2018"
schema = "nhanes_bronze"

# Create schema if it does not exist
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}"
)

# ----------------------------------------
# 2. Find all XPT files
# ----------------------------------------

xpt_files = sorted(source_folder.glob("*.xpt"))

print(f"Found {len(xpt_files)} XPT files.")

# ----------------------------------------
# 3. Convert XPT files into Delta tables
# ----------------------------------------

conversion_log = []

for file_path in xpt_files:

    try:
        print(f"Processing {file_path.name}...")

        # Read SAS Transport file
        pdf, metadata = pyreadstat.read_xport(
            str(file_path)
        )

        # Clean table name
        table_name = Path(file_path.name).stem.lower()
        table_name = re.sub(
            r"[^a-z0-9_]",
            "_",
            table_name
        )

        # Convert pandas DataFrame to Spark DataFrame
        sdf = spark.createDataFrame(pdf)

        # Fully qualified Delta table name
        full_table_name = (
            f"{catalog}.{schema}.{table_name}"
        )

        # Write as Delta table
        (
            sdf.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(full_table_name)
        )

        conversion_log.append({
            "File": file_path.name,
            "Table": full_table_name,
            "Rows": len(pdf),
            "Columns": len(pdf.columns),
            "Status": "Success"
        })

        print(f"Created Delta table: {full_table_name}")

    except Exception as error:

        conversion_log.append({
            "File": file_path.name,
            "Table": None,
            "Rows": None,
            "Columns": None,
            "Status": "Failed",
            "Error": str(error)
        })

        print(f"Failed: {file_path.name}")
        print(f"Reason: {error}")

# ----------------------------------------
# 4. Display conversion report
# ----------------------------------------

conversion_report = pd.DataFrame(conversion_log)

display(conversion_report)


Found 133 XPT files.
Processing ACQ_J.xpt...
Created Delta table: nhanes_2017___2018.nhanes_bronze.acq_j
Processing ALB_CR_J.xpt...
Created Delta table: nhanes_2017___2018.nhanes_bronze.alb_cr_j
Processing ALQ_J.xpt...
Created Delta table: nhanes_2017___2018.nhanes_bronze.alq_j
Processing AUQ_J.xpt...
Created Delta table: nhanes_2017___2018.nhanes_bronze.auq_j
Processing AUXAR_J.xpt...
Created Delta table: nhanes_2017___2018.nhanes_bronze.auxar_j
Processing AUXTYM_J.xpt...
Created Delta table: nhanes_2017___2018.nhanes_bronze.auxtym_j
Processing AUXWBR_J.xpt...
Created Delta table: nhanes_2017___2018.nhanes_bronze.auxwbr_j
Processing AUX_J.xpt...
Created Delta table: nhanes_2017___2018.nhanes_bronze.aux_j
Processing BIOPRO_J.xpt...
Created Delta table: nhanes_2017___2018.nhanes_bronze.biopro_j
Processing BMX_J.xpt...
Created Delta table: nhanes_2017___2018.nhanes_bronze.bmx_j
Processing BPQ_J.xpt...
Created Delta table: nhanes_2017___2018.nhanes_bronze.bpq_j
Processing BPXO_J.xpt...
Cr

File,Table,Rows,Columns,Status,Error
ACQ_J.xpt,nhanes_2017___2018.nhanes_bronze.acq_j,8421.0,6.0,Success,null
ALB_CR_J.xpt,nhanes_2017___2018.nhanes_bronze.alb_cr_j,7936.0,8.0,Success,null
ALQ_J.xpt,nhanes_2017___2018.nhanes_bronze.alq_j,5533.0,10.0,Success,null
AUQ_J.xpt,nhanes_2017___2018.nhanes_bronze.auq_j,8897.0,58.0,Success,null
AUXAR_J.xpt,nhanes_2017___2018.nhanes_bronze.auxar_j,33721.0,344.0,Success,null
AUXTYM_J.xpt,nhanes_2017___2018.nhanes_bronze.auxtym_j,5585.0,514.0,Success,null
AUXWBR_J.xpt,nhanes_2017___2018.nhanes_bronze.auxwbr_j,5682.0,323.0,Success,null
AUX_J.xpt,nhanes_2017___2018.nhanes_bronze.aux_j,3131.0,90.0,Success,null
BIOPRO_J.xpt,nhanes_2017___2018.nhanes_bronze.biopro_j,6401.0,41.0,Success,null
BMX_J.xpt,nhanes_2017___2018.nhanes_bronze.bmx_j,8704.0,21.0,Success,null
